In [1]:
import json
import os
import random

import h5py
import numpy as np

import time
%matplotlib inline
import matplotlib.pyplot as plt

import robosuite
import imageio
from pprint import pprint   
from lxml import etree

In [2]:
dataset_type = 'ph'

In [3]:
demo_path = f"data/demonstrations/can/{dataset_type}"
hdf5_path = os.path.join(demo_path, f"new_env_demo_{dataset_type}.hdf5")
# hdf5_path = "processed_dataset/new_dataset_ph.hdf5"
f = h5py.File(hdf5_path, "r")
# env_name = f["data"].attrs["env"]
env_name ="PickPlaceCansMilk"
env_info = json.loads(f["data"].attrs["env_info"])
env_info['camera_names'] = ['frontview', 'birdview', 'agentview', 'robot0_eye_in_hand']
# env_info["camera_names"] = "frontview"
env_info["has_renderer"] = False
env_info["use_camera_obs"] = True
env_info["ignore_done"] = True
env_info["has_offscreen_renderer"] = True
env_info["camera_heights"] = 1024
env_info["camera_widths"] = 1024

In [4]:
env = robosuite.make(env_name, **env_info)

# list of all demonstrations episodes
demos = list(f["data"].keys())

# print("Playing back random episode... (press ESC to quit)")

i = 0
ep = demos[i]
# ep = "demo_2"
# os.path.exists(ep) or os.makedirs(ep)
env.reset()

# read the model xml, using the metadata stored in the attribute for this episode
# model_xml = f["data/{}".format(ep)].attrs["model_file"]

# env.reset()
# xml = env.edit_model_xml(model_xml)
# env.reset_from_xml_string(xml)
# env.sim.reset()

states = f["data/{}/states".format(ep)][()]

# load the actions and play them back open-loop
actions = np.array(f["data/{}/actions".format(ep)][()])

# find obj_to_use
env.sim.set_state_from_flattened(states[-2])
env.step(actions[-1])
obj_id = np.where(env.objects_in_bins==1)[0][0]
obj_to_use = env.objects[obj_id]
obj_name = env.obj_names[obj_id]

env.sim.set_state_from_flattened(states[0])
env.sim.forward()

num_actions = actions.shape[0]
keyframes = []
keyframe_infos = []


In [15]:
collisions = 0
prev_image = None
same = True
for state in states:
    env.sim.set_state_from_flattened(state)
    env.sim.forward()
    # env.sim.step()
    env._update_observables()
    # # Pos
    # obj_geom_id = list(env.obj_geom_id.values())[obj_id][0]
    # print(env.sim.data.geom_xpos[obj_geom_id])
    # # contacts
    # contacts = env.get_contacts(obj_to_use)
    # print(contacts)
    if prev_image is not None:
        same = same and np.all(np.equal(env._get_observations()['frontview_image'], prev_image))
    # print(env._get_observations()['frontview_image'])
    prev_image = env._get_observations()['frontview_image']
same

False

In [6]:
env._observables.keys()

odict_keys(['robot0_joint_pos', 'robot0_joint_pos_cos', 'robot0_joint_pos_sin', 'robot0_joint_vel', 'robot0_eef_pos', 'robot0_eef_quat', 'robot0_eef_vel_lin', 'robot0_eef_vel_ang', 'robot0_gripper_qpos', 'robot0_gripper_qvel', 'frontview_image', 'birdview_image', 'agentview_image', 'robot0_eye_in_hand_image', 'world_pose_in_gripper', 'Milk_pos', 'Milk_quat', 'Milk_to_robot0_eef_pos', 'Milk_to_robot0_eef_quat', 'Bread_pos', 'Bread_quat', 'Bread_to_robot0_eef_pos', 'Bread_to_robot0_eef_quat', 'Cereal_pos', 'Cereal_quat', 'Cereal_to_robot0_eef_pos', 'Cereal_to_robot0_eef_quat', 'Can_pos', 'Can_quat', 'Can_to_robot0_eef_pos', 'Can_to_robot0_eef_quat', 'Lemon_pos', 'Lemon_quat', 'Lemon_to_robot0_eef_pos', 'Lemon_to_robot0_eef_quat', 'Bottle_pos', 'Bottle_quat', 'Bottle_to_robot0_eef_pos', 'Bottle_to_robot0_eef_quat'])

In [58]:
env.obj_names[obj_id]

'Milk'

In [7]:
print('\n'.join(env._observables.keys()))

robot0_joint_pos
robot0_joint_pos_cos
robot0_joint_pos_sin
robot0_joint_vel
robot0_eef_pos
robot0_eef_quat
robot0_eef_vel_lin
robot0_eef_vel_ang
robot0_gripper_qpos
robot0_gripper_qvel
frontview_image
birdview_image
agentview_image
robot0_eye_in_hand_image
world_pose_in_gripper
Milk_pos
Milk_quat
Milk_to_robot0_eef_pos
Milk_to_robot0_eef_quat
Bread_pos
Bread_quat
Bread_to_robot0_eef_pos
Bread_to_robot0_eef_quat
Cereal_pos
Cereal_quat
Cereal_to_robot0_eef_pos
Cereal_to_robot0_eef_quat
Can_pos
Can_quat
Can_to_robot0_eef_pos
Can_to_robot0_eef_quat
Lemon_pos
Lemon_quat
Lemon_to_robot0_eef_pos
Lemon_to_robot0_eef_quat
Bottle_pos
Bottle_quat
Bottle_to_robot0_eef_pos
Bottle_to_robot0_eef_quat


In [17]:
f.close()


In [ ]:
env.close()

## Distance

In [4]:
for j, action in enumerate(actions):
    obs, reward, done, info = env.step(action)

    obj_geom_id = list(env.obj_geom_id.values())[obj_id][0]
    print(env.sim.data.geom_xpos[obj_geom_id])

    if j < num_actions - 1:
        # ensure that the actions deterministically lead to the same recorded states
        state_playback = env.sim.get_state().flatten()
        state_data = states[j + 1]
        if not np.all(np.equal(state_data, state_playback)):
            err = np.linalg.norm(state_data - state_playback)
            # if err > 2:
            #     print("State mismatch in demo {} at step {}! Error: {}".format(ep, j, err))
            #     break
            # print(f"[warning] playback diverged by {err:.2f} for ep {ep} at step {j}")

[ 0.11122177 -0.21756416  0.86040493]
[ 0.11121268 -0.21757465  0.86040768]
[ 0.11122034 -0.21756338  0.86040963]
[ 0.11123176 -0.21756777  0.86041437]
[ 0.11123936 -0.21757273  0.86040684]
[ 0.11121131 -0.21757966  0.86040739]
[ 0.11120132 -0.21757811  0.86040883]
[ 0.11124176 -0.21758652  0.86040999]
[ 0.1112469  -0.2175935   0.86040966]
[ 0.11122955 -0.2175567   0.86041145]
[ 0.11123081 -0.21755635  0.86041447]
[ 0.11120685 -0.21759053  0.86041197]
[ 0.11124386 -0.21757724  0.86041136]
[ 0.11121438 -0.21759268  0.8604117 ]
[ 0.11124648 -0.21758705  0.86040948]
[ 0.11124976 -0.21755467  0.86041306]
[ 0.11126383 -0.21758944  0.86041523]
[ 0.11125142 -0.21757189  0.86041417]
[ 0.11122178 -0.21755395  0.86041259]
[ 0.11120328 -0.21757024  0.86041192]
[ 0.11119846 -0.21756147  0.86041298]
[ 0.11120968 -0.21755475  0.8604143 ]
[ 0.11121723 -0.21755466  0.86041402]
[ 0.11120251 -0.21756645  0.86041247]
[ 0.11120067 -0.21756412  0.86041309]
[ 0.11121012 -0.2175908   0.86041184]
[ 0.11124258

In [5]:
len(actions)

722

In [6]:
env.sim.set_state_from_flattened(states[-2])
env.step(actions[-1])
obj_id = np.where(env.objects_in_bins==1)[0][0]
obj_to_use = env.objects[obj_id]

env.sim.set_state_from_flattened(states[350])
env.sim.forward()
obj_geom_id = list(env.obj_geom_id.values())[obj_id][0]

obj_pos = env.sim.data.geom_xpos[obj_geom_id]
for geoms in env.obj_geom_id.values():
    if geoms[0] == obj_geom_id:
        continue
    pos = env.sim.data.geom_xpos[geoms[0]]
    dist = np.linalg.norm(pos - obj_pos)
    print(dist)

0.12423491231881739
0.12135332733080982
0.12679644009997634
0.0538627958894629
0.14519654457877768


In [7]:
env.bin1_pos, env.bin2_pos, obj_pos

(array([ 0.1 , -0.25,  0.8 ]),
 array([0.1 , 0.28, 0.8 ]),
 array([ 0.11231349, -0.21983659,  0.86010401]))

## Speed

In [8]:
env.sim.data.get_geom_xvelp('gripper0_hand_collision')

array([6.75036944e-05, 4.75720273e-03, 7.99432476e-03])

In [9]:
env.sim.set_state_from_flattened(states[0])
env.sim.forward()

for j, action in enumerate(actions):
    obs, reward, done, info = env.step(action)

    obj_geom_id = list(env.obj_geom_id.values())[obj_id][0]
    print(env.sim.data.get_geom_xvelp('gripper0_hand_collision'))

    if j < num_actions - 1:
        # ensure that the actions deterministically lead to the same recorded states
        state_playback = env.sim.get_state().flatten()
        state_data = states[j + 1]
        if not np.all(np.equal(state_data, state_playback)):
            err = np.linalg.norm(state_data - state_playback)
            # if err > 2:
            #     print("State mismatch in demo {} at step {}! Error: {}".format(ep, j, err))
            #     break
            # print(f"[warning] playback diverged by {err:.2f} for ep {ep} at step {j}")

[0.01529055 0.1491354  0.01635273]
[0.00348827 0.10530562 0.00917193]
[-0.00456086  0.05850474  0.00281599]
[-0.00919831  0.02623194 -0.00122456]
[-0.01169689  0.00379537 -0.00371621]
[-0.01300322 -0.01173748 -0.00531326]
[-0.01374125 -0.02210121 -0.00641029]
[-0.0142368  -0.02843544 -0.00721339]
[-0.01458823 -0.03158585 -0.00780202]
[-0.01476216 -0.0322626  -0.00818396]
[-0.01468267 -0.03109969 -0.00834205]
[-0.01428236 -0.02866901 -0.00826014]
[-0.01353052 -0.02545905 -0.0079377 ]
[-0.01243682 -0.02186452 -0.00739175]
[-0.01114726 -0.017741   -0.00668543]
[-0.00966    -0.01421027 -0.00586314]
[-0.00800385 -0.01096381 -0.00493538]
[-0.0062615  -0.00804593 -0.00394517]
[-0.00466674 -0.00581063 -0.00300582]
[-0.00329756 -0.00442921 -0.00217173]
[-0.00252072 -0.00330811 -0.00169866]
[-0.00199796 -0.00237799 -0.00137676]
[-0.00157055 -0.00170294 -0.00110606]
[-0.00122397 -0.00124886 -0.00088041]
[-0.00095876 -0.00097275 -0.00070135]
[-0.00077261 -0.00078241 -0.0005681 ]
[-0.00065429 -0.00

array([ 5.89018274e-05, -1.33653727e-04,  3.58399591e-04])

In [ ]:
list(env.object_to_id.keys())

['milk', 'bread', 'cereal', 'can', 'lemon', 'bottle']